# Train YOLO26n-seg on the stool dataset (Colab)

**Before running anything:** go to **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.
Then run the cells top to bottom with the ▶ button (or Shift+Enter). Each cell waits for the previous one.

What you get at the end:
- `best.pt` – PyTorch weights (for Python / retraining)
- `best.onnx` – for the website / phone with `onnxruntime-web`

**Dataset license:** the "stool classification" dataset by *smartgut* on
[Roboflow Universe](https://universe.roboflow.com/smartgut/stool-classification-uvnqx) is licensed
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). You may use and modify it, including commercially,
as long as you credit the source — see `DATASET_ATTRIBUTION.md` and put the same credit on the website/app.

**Disclaimer (put this on the website/app too):** This is an informational/educational Bristol Stool Scale demo, **not medical advice**
and not a medical device. Predictions may be wrong; consult a healthcare professional for health concerns. Full text in `DATASET_ATTRIBUTION.md`.

In [ ]:
# 1) Confirm the GPU is on. You should see "Tesla T4" (or similar). If it says "command not found", fix the runtime type (see above).
!nvidia-smi -L

In [ ]:
# 2) Install Ultralytics (YOLO26 lives here) + ONNX tooling. Takes ~1 min.
%pip install -q -U ultralytics onnx onnxruntime onnxslim
import ultralytics
ultralytics.checks()

## 3) Upload the dataset

Run the next cell, click **Choose Files**, and pick `stool_yolo_seg.zip` from your `hackthevalley` folder
(it's the already-converted YOLO version of your COCO export, 15 MB).

*Alternative if the upload is slow:* put the zip in Google Drive, then replace this cell with
`from google.colab import drive; drive.mount('/content/drive')` and use `/content/drive/MyDrive/stool_yolo_seg.zip` below.

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick stool_yolo_seg.zip
zip_path = next(iter(uploaded))    # name of the file you uploaded

In [ ]:
# 4) Unzip + point data.yaml at the Colab path, then sanity check.
import os, zipfile, glob
os.makedirs('/content/dataset', exist_ok=True)
zipfile.ZipFile(zip_path).extractall('/content/dataset')
!sed -i 's|^path: .*|path: /content/dataset|' /content/dataset/data.yaml
print(open('/content/dataset/data.yaml').read())
for s in ['train', 'valid', 'test']:
    print(s, len(glob.glob(f'/content/dataset/images/{s}/*')), 'images')

## 5) Train

- `yolo26n-seg.pt` = YOLO26 **n**ano **seg**mentation, pretrained on COCO. Smallest/fastest, right choice for a phone.
- `imgsz=320`: your images are 312×312, so 320 (next multiple of 32) loses nothing and runs ~4× faster on a phone than 640.
- `epochs=100` with `patience=20` stops early if validation stops improving. Expect **~15–30 min** on a T4.

Watch the `mAP50(M)` column (mask accuracy). If Colab disconnects, re-run cells 1–4 and set `resume=True` with `model = YOLO('runs/segment/stool/weights/last.pt')`.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n-seg.pt')   # downloads the pretrained weights automatically

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=320,
    batch=32,
    patience=20,
    project='runs/segment',
    name='stool',
    exist_ok=True,
)

In [ ]:
# 6) Evaluate on the held-out test split and look at a few predictions.
best = YOLO('runs/segment/stool/weights/best.pt')
metrics = best.val(data='/content/dataset/data.yaml', split='test', imgsz=320)
print('box mAP50-95:', round(metrics.box.map, 3), '| mask mAP50-95:', round(metrics.seg.map, 3))

from IPython.display import Image, display
display(Image('runs/segment/stool/results.png', width=900))        # training curves
display(Image('runs/segment/stool/val_batch0_pred.jpg', width=900))  # predictions on val images

## 7) Export to ONNX for the browser

- `nms=False` → YOLO26's **end-to-end head**: the model outputs final detections, so you don't write NMS in JavaScript.
- `imgsz=320`, `dynamic=False`, fp32 → fixed input `[1, 3, 320, 320]`, which is what `onnxruntime-web` likes.
- `opset=17` is supported by current `onnxruntime-web` (WASM and WebGPU backends).

In [ ]:
onnx_path = best.export(format='onnx', imgsz=320, nms=False, dynamic=False, half=False, simplify=True, opset=17)
print('exported:', onnx_path)

# Print the input/output tensor shapes – you need these for the JavaScript side.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
for i in sess.get_inputs():  print('INPUT ', i.name, i.shape, i.type)
for o in sess.get_outputs(): print('OUTPUT', o.name, o.shape, o.type)

In [ ]:
# 8) Download both files to your computer (also saved under runs/segment/stool/weights/ in this session).
from google.colab import files
files.download('runs/segment/stool/weights/best.pt')
files.download(onnx_path)

## Using `best.onnx` on the website (phone)

Load with [`onnxruntime-web`](https://onnxruntime.ai/docs/tutorials/web/) (`npm i onnxruntime-web`, or the CDN script).

**Preprocess** the camera frame: draw it onto a 320×320 canvas (letterbox or stretch – the training data was stretched to square, so stretching is fine),
read RGB pixels, divide by 255, and pack as **NCHW float32** `[1, 3, 320, 320]` (all R values, then all G, then all B).

**Outputs** (printed by cell 7 above; expected shapes):

| output | shape | meaning |
|---|---|---|
| `output0` | `[1, 300, 38]` | per detection: `x1, y1, x2, y2, score, class_id, 32 mask coefficients` (pixel coords in the 320 input) |
| `output1` | `[1, 32, 80, 80]` | mask prototypes |

Steps per frame: keep rows with `score > 0.25`; `class_id` 0–6 = Bristol type 1–7 (`names` in data.yaml).
For the mask of a detection: `mask[y][x] = sigmoid( Σ_k coeff[k] * proto[k][y][x] )` over the 80×80 grid, threshold at 0.5,
crop to the box (divide box coords by 4), and scale up to the display size. If you only need the class + bounding box, skip the mask entirely.

```js
import * as ort from 'onnxruntime-web';
const session = await ort.InferenceSession.create('best.onnx', { executionProviders: ['webgpu', 'wasm'] });
const input = new ort.Tensor('float32', nchwFloat32Array, [1, 3, 320, 320]);
const { output0, output1 } = await session.run({ images: input });
const [, nDet, stride] = output0.dims;        // 300, 38
for (let i = 0; i < nDet; i++) {
  const row = output0.data.subarray(i * stride, (i + 1) * stride);
  const [x1, y1, x2, y2, score, cls] = row;
  if (score < 0.25) continue;
  const coeffs = row.subarray(6);             // 32 mask coefficients
  // ... draw box, compute mask from output1 if needed
}
```